# 04 — Assign topics to SelfAware

SelfAware ships without topic labels. This notebook classifies every question into a fixed taxonomy with Qwen2.5-3B-Instruct and writes the labels back into `data/selfaware/SelfAware.json` as a `topic` field on every example — which `load_selfaware` then surfaces automatically.

**Inputs:** `data/selfaware/SelfAware.json` (raw, no `topic` field).
**Outputs:** the same file, enriched in place with a `topic` field.

Required before **Day 3** (cross-topic generalization) — the held-out-topic split has nothing to split on until this runs. ~3.4k short classifications: a few minutes on a Colab GPU, much slower on CPU.

In [ ]:
# --- Colab setup (skip these two lines when running locally) ---
# from google.colab import drive; drive.mount('/content/drive')
# %cd "/content/drive/MyDrive/abstention-geometry/notebooks"
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [ ]:
import json

import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

## Topic taxonomy

A fixed, mutually-exclusive list keeps the Day 3 train/test split clean. `other` catches anything the classifier can't place.

In [ ]:
import src

# Resolve the data path from the src package location, not the working
# directory — robust whether or not the Colab %cd above succeeded.
REPO_ROOT = Path(src.__file__).resolve().parent.parent
DATA_PATH = str(REPO_ROOT / 'data' / 'selfaware' / 'SelfAware.json')
JUDGE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

TOPICS = [
    'history',
    'science & nature',
    'geography',
    'sports & games',
    'arts & entertainment',
    'politics & law',
    'literature & language',
    'technology',
    'society & culture',
    'religion & philosophy',
    'health & medicine',
    'business & economics',
]
print('DATA_PATH =', DATA_PATH)

## Load raw questions

In [ ]:
raw = json.load(open(DATA_PATH))
questions = [ex['question'] for ex in raw['example']]
print(len(questions), 'questions')
print('already has a topic field:', 'topic' in raw['example'][0])

## Load the Qwen classifier

Same model as the Day 1 judge. `padding_side='left'` is required for correct batched generation with a decoder-only model.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'
clf = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, torch_dtype=torch.bfloat16, device_map=device
)
clf.eval()
print('classifier loaded on', device)

## Classify every question

Greedy decoding, one topic per question. The reply is matched back to the taxonomy: first by full-name substring, then by the topic's leading word (so a bare `science` still maps to `science & nature`). Anything unmatched falls back to `other`.

In [ ]:
SYSTEM = (
    'You label trivia questions by subject area. Reply with exactly one '
    'topic from the list the user gives you, copied verbatim, and nothing else.'
)


def _prompt(question):
    msgs = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f"Topics: {', '.join(TOPICS)}\n\n"
                                    f"Question: {question}\n\nTopic:"},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def _match(text):
    """Map a free-text reply to a taxonomy entry, or 'other'."""
    t = text.strip().lower()
    for topic in TOPICS:
        if topic in t:
            return topic
    for topic in TOPICS:
        if topic.split()[0] in t:
            return topic
    return 'other'


@torch.no_grad()
def classify(questions, batch_size=16):
    labels = []
    for i in tqdm(range(0, len(questions), batch_size), desc='classifying'):
        batch = questions[i:i + batch_size]
        enc = tok(
            [_prompt(q) for q in batch],
            return_tensors='pt', padding=True, truncation=True, max_length=512,
        ).to(device)
        gen = clf.generate(
            **enc, max_new_tokens=10, do_sample=False, pad_token_id=tok.pad_token_id
        )
        dec = tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        labels.extend(_match(text) for text in dec)
    return labels


topics = classify(questions)
print(pd.Series(topics).value_counts())

## Write the topic-enriched JSON

Written back to the same path the rest of the pipeline reads. The raw file is recoverable from the SelfAware submodule's git history if you ever need it back.

In [ ]:
assert len(topics) == len(raw['example'])
for ex, topic in zip(raw['example'], topics):
    ex['topic'] = topic
with open(DATA_PATH, 'w') as f:
    json.dump(raw, f, indent=2, ensure_ascii=False)
print('wrote', DATA_PATH)

# Verify through the loader the rest of the pipeline uses.
from src.data import load_selfaware

df = load_selfaware(DATA_PATH)
print('\ntopic distribution as seen by load_selfaware:')
print(df['topic'].value_counts())